# 🗺️ Notebook 05 — Geospatial Analysis

**Project:** Nigeria Disease Surveillance Dashboard  
**Purpose:** Spatial analysis of disease burden across Nigeria's 37 states.

---
**What we analyse:**
1. Choropleth maps — disease incidence by state
2. Health facility overlay — access vs. burden
3. Moran's I — does high burden cluster spatially?
4. Facility accessibility gap analysis
5. Composite Disease Burden Index (DBI)
6. Spatial time-series — how burden shifts over years

> **Requires:** State shapefiles in `data/shapefiles/nigeria_states.shp`

## 1. Setup

In [ ]:
import sys, warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import folium
from IPython.display import display

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import Paths, Diseases
from src.utils.state_maps import CANONICAL_STATES, GEOPOLITICAL_ZONES
from src.etl.extract import extract_shapefiles, extract_health_facilities
from src.analysis.geospatial import (
    prepare_choropleth,
    geodataframe_to_geojson,
    compute_morans_i,
    analyse_facility_accessibility,
    accessibility_to_dataframe,
    compute_burden_index,
)

# Load master data
df = pd.read_csv(
    Paths.processed / 'master_surveillance.csv',
    parse_dates=['report_date']
)
df['year'] = df['report_date'].dt.year

# Load shapefiles
shapefiles = extract_shapefiles()
states_gdf = shapefiles.get('states')

# Load health facilities
facilities_df = extract_health_facilities()

# Load population
pop_df = pd.read_csv(Paths.processed / 'population_clean.csv') \
         if (Paths.processed / 'population_clean.csv').exists() \
         else pd.DataFrame(columns=['state','population'])

print(f'✅ Data loaded')
print(f'   Surveillance rows : {len(df):,}')
print(f'   State geometries  : {len(states_gdf) if states_gdf is not None else "NOT LOADED"}')
print(f'   Health facilities : {len(facilities_df):,}')
print(f'   Population states : {len(pop_df)}')

## 2. State Burden Aggregation

Aggregate confirmed cases and incidence per state for a chosen  
disease and year — this feeds all the map layers below.

In [ ]:
# ── Configuration — change these to explore different views ─────
DISEASE = Diseases.CHOLERA
YEAR    = 2023
# ─────────────────────────────────────────────────────────────────

state_burden = (
    df[
        (df['disease'] == DISEASE) &
        (df['year'] == YEAR)
    ]
    .groupby('state')
    .agg(
        total_cases           = ('confirmed_cases',    'sum'),
        total_deaths          = ('deaths',             'sum'),
        avg_incidence_per_100k = ('incidence_per_100k', 'mean'),
        avg_cfr_pct           = ('cfr_pct',            'mean'),
    )
    .round(3)
    .reset_index()
)

state_burden['zone'] = state_burden['state'].map(GEOPOLITICAL_ZONES)

print(f'State burden for {DISEASE} — {YEAR}')
print(f'  States with data : {len(state_burden)}/37')
print(f'  Total cases      : {state_burden["total_cases"].sum():,}')
print(f'  Highest burden   : {state_burden.nlargest(1,"total_cases")["state"].iloc[0]}')
print()
display(state_burden.nlargest(10, 'total_cases'))

## 3. Choropleth Map

Interactive Folium map showing disease incidence by state.  
Hover over each state to see case counts and incidence rate.

In [ ]:
if states_gdf is None:
    print('⚠️  State shapefiles not found.')
    print('   Download from grid3.org and place in data/shapefiles/')
else:
    # Merge burden with geometries
    choropleth_gdf = prepare_choropleth(
        burden_df      = state_burden,
        states_gdf     = states_gdf,
        value_col      = 'avg_incidence_per_100k',
    )

    # Build Folium choropleth
    m = folium.Map(
        location   = [9.0, 8.0],
        zoom_start = 6,
        tiles      = 'CartoDB positron',
    )

    # Convert to GeoJSON for the choropleth layer
    geojson_dict = geodataframe_to_geojson(
        choropleth_gdf,
        properties=['state','total_cases','avg_incidence_per_100k'],
    )

    folium.Choropleth(
        geo_data       = geojson_dict,
        data           = state_burden,
        columns        = ['state','avg_incidence_per_100k'],
        key_on         = 'feature.properties.state',
        fill_color     = 'YlOrRd',
        fill_opacity   = 0.75,
        line_opacity   = 0.4,
        line_color     = 'white',
        legend_name    = f'{DISEASE} — Incidence per 100k ({YEAR})',
        nan_fill_color = '#E8E8E8',
    ).add_to(m)

    # Tooltip on hover
    folium.GeoJson(
        geojson_dict,
        tooltip=folium.GeoJsonTooltip(
            fields   = ['state','total_cases','avg_incidence_per_100k'],
            aliases  = ['State','Total Cases','Incidence /100k'],
            localize = True,
        ),
        style_function=lambda x: {
            'fillColor':'transparent','color':'transparent','weight':0
        },
    ).add_to(m)

    # Save map
    map_path = Paths.processed / f'choropleth_{DISEASE.lower().replace(" ","_")}_{YEAR}.html'
    m.save(str(map_path))
    print(f'✅ Choropleth map saved: {map_path.name}')
    print('   Open in browser to view interactively')
    display(m)

## 4. Health Facility Overlay

Add facility locations as a separate layer on the map.

In [ ]:
if states_gdf is None:
    print('⚠️  Shapefiles required for this section')
elif facilities_df.empty:
    print('⚠️  No facility data available — download from data.humdata.org')
else:
    m2 = folium.Map(location=[9.0,8.0], zoom_start=6, tiles='CartoDB positron')

    # Add choropleth base layer
    folium.Choropleth(
        geo_data     = geojson_dict,
        data         = state_burden,
        columns      = ['state','total_cases'],
        key_on       = 'feature.properties.state',
        fill_color   = 'YlOrRd',
        fill_opacity = 0.6,
        legend_name  = f'{DISEASE} Total Cases ({YEAR})',
        nan_fill_color='#E8E8E8',
    ).add_to(m2)

    # Add facility markers
    fac_group = folium.FeatureGroup(name='Health Facilities')
    lat_col = next((c for c in facilities_df.columns if 'lat' in c.lower()), None)
    lon_col = next((c for c in facilities_df.columns if 'lon' in c.lower() or 'lng' in c.lower()), None)

    if lat_col and lon_col:
        valid_fac = facilities_df.dropna(subset=[lat_col, lon_col]).head(2000)
        for _, row in valid_fac.iterrows():
            folium.CircleMarker(
                location     = [float(row[lat_col]), float(row[lon_col])],
                radius       = 3,
                color        = '#185FA5',
                fill         = True,
                fill_opacity = 0.6,
                tooltip      = str(row.get('facility_name', 'Facility')),
            ).add_to(fac_group)

        fac_group.add_to(m2)
        folium.LayerControl().add_to(m2)

        print(f'✅ {len(valid_fac):,} facility markers added')
        display(m2)
    else:
        print('⚠️  Could not identify lat/lon columns in facilities data')

## 5. Moran's I — Spatial Autocorrelation

Tests whether high-burden states cluster spatially (positive I)  
or are dispersed (negative I).  

> **Requires:** `libpysal` and `esda` — install with `pip install libpysal esda`

In [ ]:
if states_gdf is None:
    print('⚠️  Shapefiles required for Moran\'s I')
else:
    print(f'Computing Moran\'s I for all diseases ({YEAR})...\n')

    for disease in df['disease'].unique():
        burden = (
            df[(df['disease']==disease) & (df['year']==YEAR)]
            .groupby('state')
            .agg(total_cases=('confirmed_cases','sum'))
            .reset_index()
        )
        result = compute_morans_i(
            burden_df  = burden,
            states_gdf = states_gdf,
            disease    = disease,
            value_col  = 'total_cases',
            year       = YEAR,
        )
        sig = '⭐' if result.significant else '  '
        print(f'  {sig} {disease:<20} '
              f'I={result.morans_i:+.4f}  '
              f'p={result.p_value:.4f}  '
              f'pattern={result.pattern}')

    print()
    print('  ⭐ = statistically significant spatial clustering/dispersion')

## 6. Facility Accessibility Gap Analysis

In [ ]:
if facilities_df.empty or pop_df.empty:
    print('⚠️  Facility and population data required for accessibility analysis')
else:
    print(f'Analysing facility accessibility ({DISEASE})...\n')

    results = analyse_facility_accessibility(
        burden_df     = state_burden,
        facilities_df = facilities_df,
        population_df = pop_df,
        disease       = DISEASE,
    )
    access_df = accessibility_to_dataframe(results)

    import plotly.express as px
    fig = px.scatter(
        access_df,
        x        = 'facilities_per_100k',
        y        = 'disease_burden',
        color    = 'flag',
        size     = 'access_gap_score',
        text     = 'state',
        color_discrete_map = {
            'CRITICAL':'#E24B4A','POOR':'#EF9F27',
            'ADEQUATE':'#1D9E75','GOOD':'#185FA5'
        },
        title    = f'Facility Access vs Disease Burden — {DISEASE} ({YEAR})',
        labels   = {
            'facilities_per_100k':'Facilities per 100k',
            'disease_burden':'Avg Incidence /100k',
        },
        template = 'plotly_white',
    )
    # States in top-right quadrant (high burden, few facilities) are priority
    fig.add_annotation(
        x=access_df['facilities_per_100k'].quantile(0.25),
        y=access_df['disease_burden'].quantile(0.75),
        text='Priority: high burden,<br>low access',
        showarrow=False, font=dict(color='#E24B4A', size=11)
    )
    fig.update_traces(textposition='top center', textfont_size=8)
    fig.update_layout(height=480)
    fig.show()

    critical = access_df[access_df['flag']=='CRITICAL']
    print(f'  CRITICAL states ({len(critical)}): {", ".join(critical["state"].tolist())}')

## 7. Composite Disease Burden Index

In [ ]:
print(f'Computing Disease Burden Index ({YEAR})...\n')

dbi_df = compute_burden_index(df, year=YEAR)

if dbi_df.empty:
    print('⚠️  Not enough data for DBI calculation')
else:
    import plotly.express as px
    fig = px.bar(
        dbi_df.head(15).sort_values('burden_index'),
        x           = 'burden_index',
        y           = 'state',
        orientation = 'h',
        color       = 'burden_index',
        color_continuous_scale='Reds',
        title       = f'Disease Burden Index — Top 15 States ({YEAR})',
        labels      = {'burden_index':'Burden Index (0–1)','state':''},
        template    = 'plotly_white',
    )
    fig.update_layout(
        height=420,
        coloraxis_showscale=False
    )
    fig.show()

    print(f'  Top 5 highest burden states ({YEAR}):')
    for _, row in dbi_df.head(5).iterrows():
        print(f'    {row["rank"]}. {row["state"]:<20} DBI={row["burden_index"]:.3f}')

## 8. Spatial Trend — Year-on-Year Shift

Which states are getting better or worse over time?

In [ ]:
# Compare two years — 2019 vs 2023
YEAR_A, YEAR_B = 2019, 2023

def get_annual_incidence(df, disease, year):
    return (
        df[(df['disease']==disease) & (df['year']==year)]
        .groupby('state')['incidence_per_100k']
        .mean()
        .rename(f'incidence_{year}')
    )

yr_a = get_annual_incidence(df, DISEASE, YEAR_A)
yr_b = get_annual_incidence(df, DISEASE, YEAR_B)

trend_shift = pd.concat([yr_a, yr_b], axis=1).dropna()
trend_shift['change_pct'] = (
    (trend_shift[f'incidence_{YEAR_B}'] - trend_shift[f'incidence_{YEAR_A}'])
    / (trend_shift[f'incidence_{YEAR_A}'].replace(0, np.nan)) * 100
).round(1)
trend_shift = trend_shift.reset_index()
trend_shift['direction'] = trend_shift['change_pct'].apply(
    lambda x: 'Worsening' if x > 0 else 'Improving'
)

import plotly.express as px
fig = px.bar(
    trend_shift.sort_values('change_pct', ascending=False),
    x='state', y='change_pct', color='direction',
    color_discrete_map={'Worsening':'#E24B4A','Improving':'#1D9E75'},
    title=f'{DISEASE} — Incidence Change ({YEAR_A}→{YEAR_B}) by State',
    labels={'change_pct':'% Change in Incidence','state':'State'},
    template='plotly_white',
)
fig.update_layout(height=380, xaxis_tickangle=-45)
fig.show()

## 9. Summary

In [ ]:
from datetime import datetime
print('='*55)
print('  NOTEBOOK 05 — GEOSPATIAL SUMMARY')
print('='*55)
print(f'  Timestamp  : {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print(f'  Disease    : {DISEASE}')
print(f'  Year       : {YEAR}')
print()
print('  Key spatial findings (fill in after running):')
print('  • Spatial pattern: (clustered/dispersed/random)')
print('  • Moran\'s I significance: (p-value)')
print('  • CRITICAL accessibility states: (list)')
print('  • Highest DBI state: (state name)')
print()
print('  Map files saved to data/processed/')
print()
print('  ➡️  Next: Notebook 06 — Forecasting')
print('='*55)